# Notebook 13 — Batch Screening

**PharmaLens**

---

## Objective

Demonstrate the practical application of PharmaLens: **batch screening** of drug–target combinations.

Given a list of candidate drugs and protein targets, predict and rank all pairwise interactions. This is how PharmaLens would be used in a real drug discovery workflow.

In [ ]:
import os, sys, warnings, pickle
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.utils.config import *
from src.models.xgboost_model import XGBoostDTI
from src.features.drug_features import get_drug_feature_vector
from src.features.protein_features import get_protein_feature_vector

set_seed(42)
COLORS = set_plot_style()
ensure_dirs()
print("Setup complete ✓")

In [ ]:
# Load best model
model = XGBoostDTI()
model_path = XGBOOST_MODELS_DIR / 'xgb_tuned_best.json'
if not model_path.exists():
    model_path = XGBOOST_MODELS_DIR / 'xgb_best.json'
model.load(model_path)
print(f"Model loaded: {model_path}")

## 1. Define Screening Panel

Select a subset of drugs and targets for screening demonstration.

In [ ]:
# Load the clean dataset for real SMILES and sequences
df = pd.read_csv(PROCESSED_DATA_DIR / 'kiba_clean.csv')

# Select top 20 drugs (by number of interactions) and top 10 targets
top_drugs = df['drug_id'].value_counts().head(20).index.tolist()
top_targets = df['target_id'].value_counts().head(10).index.tolist()

drug_smiles = dict(zip(df['drug_id'], df['smiles']))
target_seqs = dict(zip(df['target_id'], df['sequence']))

print(f"Screening panel: {len(top_drugs)} drugs × {len(top_targets)} targets = {len(top_drugs)*len(top_targets)} combinations")

In [ ]:
# Compute features for the screening panel
drug_fv = {}
for did in top_drugs:
    fv = get_drug_feature_vector(drug_smiles[did])
    if fv is not None:
        drug_fv[did] = fv

target_fv = {}
for tid in top_targets:
    fv = get_protein_feature_vector(target_seqs[tid])
    target_fv[tid] = fv

print(f"Drug features computed: {len(drug_fv)}")
print(f"Target features computed: {len(target_fv)}")

## 2. Batch Prediction

In [ ]:
# Predict all combinations
results = []
for did in drug_fv:
    for tid in target_fv:
        combined = np.concatenate([drug_fv[did], target_fv[tid]])
        pred = model.predict(combined.reshape(1, -1))[0]
        results.append({
            'drug_id': did,
            'target_id': tid,
            'predicted_score': round(float(pred), 4),
        })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('predicted_score', ascending=True)  # Lower = stronger binding
results_df['rank'] = range(1, len(results_df) + 1)

print(f"\nScreening complete: {len(results_df)} predictions")
print(f"\nTop 20 Predicted Interactions (strongest binding):")
print(results_df.head(20).to_string(index=False))

In [ ]:
# Heatmap of predictions
pivot = results_df.pivot(index='drug_id', columns='target_id', values='predicted_score')

fig, ax = plt.subplots(figsize=(14, 10))
im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlBu_r', interpolation='nearest')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([str(x)[:15] for x in pivot.columns], rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([str(x)[:15] for x in pivot.index], fontsize=8)
ax.set_xlabel('Protein Target')
ax.set_ylabel('Drug')
ax.set_title('Predicted Interaction Scores (Screening Panel)', fontweight='bold')
plt.colorbar(im, label='Predicted KIBA Score', shrink=0.8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'screening_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Save screening results
results_df.to_csv(TABLES_DIR / 'screening_results.csv', index=False)
print(f"Results saved: {TABLES_DIR / 'screening_results.csv'}")

## Observations

1. **Ranking** allows prioritization of the most promising drug-target pairs for experimental validation.

2. **The heatmap** shows interaction patterns across the screening panel — some targets may bind strongly with many drugs (promiscuous targets), while others are more selective.

3. **This workflow** demonstrates the practical utility of PharmaLens: given new candidate molecules, predict and rank their interactions with protein targets of interest.

> **Reminder:** These are computational predictions. Top-ranked pairs should be validated experimentally.

**Next:** [Notebook 14 — Final Evaluation](./14_final_evaluation.ipynb)